In [2]:
# Import Libraries

import tensorflow as tf
import tensorflow.keras as keras
import numpy as np
from tensorflow.keras import layers
from tensorflow.keras import regularizers
import gc
import golois
import sys

print("Python version :"+ sys.version)
print ("Tensorflow version ", tf.__version__)
print("GPU disponible:", tf.config.list_physical_devices('GPU'))

Python version :3.9.21 (main, Mar 18 2025, 15:05:54) 
[Clang 16.0.0 (clang-1600.0.26.6)]
Tensorflow version  2.15.0
GPU disponible: [PhysicalDevice(name='/physical_device:GPU:0', device_type='GPU')]


In [3]:
# Configuration

planes = 31
moves = 361
N = 10000
batch = 128


input_data = np.random.randint(2, size=(N, 19, 19, planes))
input_data = input_data.astype ('float32')

policy = np.random.randint(moves, size=(N,))
policy = keras.utils.to_categorical (policy)

value = np.random.randint(2, size=(N,))
value = value.astype ('float32')

end = np.random.randint(2, size=(N, 19, 19, 2))
end = end.astype ('float32')

groups = np.zeros((N, 19, 19, 1))
groups = groups.astype ('float32')

In [4]:
# Get Validation Data

print ("getValidation", flush = True)
golois.getValidation (input_data, policy, value, end)

getValidation


r.shape = (10000, 19, 19, 31)
nbExamples = 10000
nbPositionsSGF = 102208897
nbPositionsSGF = 102208897
loading validation.data


In [5]:
# Model Creation
#
# Architecture simple de type AlphaGo-style, avec deux têtes :
# une policy (pour prédire un coup) et une value (pour prédire l’issue de la partie).
#
# Policy - Prédit le prochain coup. Valeur entre 1 et 361
#          Utilise la Categorical Cross Entropy.
# Value  - Prédit la probabilité de victoire. Valeur entre 0 et 1


#filters = 42
#filter_size = 3
#block_iteration=5

filters = 64
filter_size = 3
block_iteration=2


l2_reg = 0.0001


def get_iteration_block(x, filters=filters, filter_size=filter_size):
  x = layers.Conv2D(filters, filter_size, activation='relu', padding='same')(x)
  return x

def get_model(filters=filters, filter_size=filter_size, l2_reg = l2_reg):

  input = keras.Input(shape=(19, 19, planes), name='board')
  x = layers.Conv2D(filters, 1, activation='relu', padding='same')(input)

  for i in range (block_iteration):
     x = get_iteration_block(x, filters, filter_size)


  policy_head = layers.Conv2D(1, 1, activation='relu', padding='same', use_bias = False, kernel_regularizer=regularizers.l2(l2_reg))(x)
  policy_head = layers.Flatten()(policy_head)
  policy_head = layers.Activation('softmax', name='policy')(policy_head)

  value_head = layers.Conv2D(1, 1, activation='relu', padding='same', use_bias = False, kernel_regularizer=regularizers.l2(l2_reg))(x)
  value_head = layers.Flatten()(value_head)
  value_head = layers.Dense(50, activation='relu', kernel_regularizer=regularizers.l2(l2_reg))(value_head)
  value_head = layers.Dense(1, activation='sigmoid', name='value', kernel_regularizer=regularizers.l2(l2_reg))(value_head)

  model = keras.Model(inputs=input, outputs=[policy_head, value_head])
  return model

model = get_model(filters, filter_size, l2_reg)
model.summary ()


Model: "model"
__________________________________________________________________________________________________
 Layer (type)                Output Shape                 Param #   Connected to                  
 board (InputLayer)          [(None, 19, 19, 31)]         0         []                            
                                                                                                  
 conv2d (Conv2D)             (None, 19, 19, 64)           2048      ['board[0][0]']               
                                                                                                  
 conv2d_1 (Conv2D)           (None, 19, 19, 64)           36928     ['conv2d[0][0]']              
                                                                                                  
 conv2d_2 (Conv2D)           (None, 19, 19, 64)           36928     ['conv2d_1[0][0]']            
                                                                                              

2025-03-31 10:19:09.845736: I metal_plugin/src/device/metal_device.cc:1154] Metal device set to: Apple M4 Pro
2025-03-31 10:19:09.845789: I metal_plugin/src/device/metal_device.cc:296] systemMemory: 48.00 GB
2025-03-31 10:19:09.845798: I metal_plugin/src/device/metal_device.cc:313] maxCacheSize: 18.00 GB
2025-03-31 10:19:09.845858: I tensorflow/core/common_runtime/pluggable_device/pluggable_device_factory.cc:306] Could not identify NUMA node of platform GPU ID 0, defaulting to 0. Your kernel may not have been built with NUMA support.
2025-03-31 10:19:09.846094: I tensorflow/core/common_runtime/pluggable_device/pluggable_device_factory.cc:272] Created TensorFlow device (/job:localhost/replica:0/task:0/device:GPU:0 with 0 MB memory) -> physical PluggableDevice (device: 0, name: METAL, pci bus id: <undefined>)


In [7]:
from tensorflow.keras import optimizers, callbacks

# Model Compilation and Train

# Epochs number
epochs = 250

# Scheduler de learning rate (comme dans le papier)
def get_learning_rate(epoch):
    if epoch < 10:
        return 0.01
    elif epoch < 50:
        return 0.005
    elif epoch < 100:
        return 0.0005
    elif epoch < 150:
        return 0.00005
    elif epoch < 200:
        return 0.000005
    else:
        return 0.0000005

# Optimiseur Adam avec taux d’apprentissage initial
optimizer = tf.keras.optimizers.legacy.Adam(learning_rate=get_learning_rate(0),
                             beta_1=0.9,
                             beta_2=0.999,
                             epsilon=1e-08,
                             amsgrad=False)

# Poids des pertes
policy_weight = 1.0
value_weight = 4.0

# Compilation du modèle
model.compile(optimizer=optimizer,
              loss={'policy': 'categorical_crossentropy', 'value': 'binary_crossentropy'},
              loss_weights={'policy': policy_weight, 'value': value_weight},
              metrics={'policy': 'categorical_accuracy', 'value': 'mse'})

# Entraînement avec scheduler
for i in range(1, epochs + 1):

    # MAJ du learning rate manuellement
    lr = get_learning_rate(i)
    keras.backend.set_value(model.optimizer.learning_rate, lr)
    print('epoch ' + str(i)+ ', lr '+str(lr))

    # Mise à jour dynamique du batch
    golois.getBatch(input_data, policy, value, end, groups, i * N)

    history = model.fit(input_data,
                        {'policy': policy, 'value': value},
                        epochs=1,
                        batch_size=batch)

    if i % 5 == 0:
        gc.collect()

    if i % epochs == 0:
        golois.getValidation(input_data, policy, value, end)
        val = model.evaluate(input_data,
                             [policy, value], verbose=0, batch_size=batch)
        print("val =", val)
        model.save('mchettih.h5')

epoch 1, lr 0.01


r.shape = (10000, 19, 19, 31)
nbExamples = 10000


79/79 [==============================] - 2s 23ms/step - loss: 8.6620 - policy_loss: 5.8889 - value_loss: 0.6933 - policy_categorical_accuracy: 0.0014 - value_mse: 0.1207
epoch 2, lr 0.01
 4/79 [>.............................] - ETA: 1s - loss: 8.6626 - policy_loss: 5.8889 - value_loss: 0.6934 - policy_categorical_accuracy: 0.0020 - value_mse: 0.1242    

r.shape = (10000, 19, 19, 31)
nbExamples = 10000


79/79 [==============================] - 2s 20ms/step - loss: 8.6625 - policy_loss: 5.8889 - value_loss: 0.6934 - policy_categorical_accuracy: 9.0000e-04 - value_mse: 0.1230
epoch 3, lr 0.01
 4/79 [>.............................] - ETA: 1s - loss: 8.6611 - policy_loss: 5.8889 - value_loss: 0.6931 - policy_categorical_accuracy: 0.0020 - value_mse: 0.1188    

...

r.shape = (10000, 19, 19, 31)
nbExamples = 10000


79/79 [==============================] - 2s 22ms/step - loss: 8.6603 - policy_loss: 5.8889 - value_loss: 0.6929 - policy_categorical_accuracy: 0.0011 - value_mse: 0.1247
epoch 25, lr 0.005
 4/79 [>.............................] - ETA: 1s - loss: 8.6629 - policy_loss: 5.8889 - value_loss: 0.6935 - policy_categorical_accuracy: 0.0000e+00 - value_mse: 0.1228

r.shape = (10000, 19, 19, 31)
nbExamples = 10000


28/79 [=========>....................] - ETA: 1s - loss: 8.6624 - policy_loss: 5.8889 - value_loss: 0.6934 - policy_categorical_accuracy: 8.3705e-04 - value_mse: 0.1212

KeyboardInterrupt: 

In [ ]:
from tensorflow.keras import optimizers, callbacks

# Model Compilation and Train

# Epochs number
epochs = 250

# Optimiseur Adadelta avec taux d’apprentissage initial
optimizer = keras.optimizers.Adadelta(
    learning_rate=1.0,     # généralement 1.0 par défaut
    rho=0.95,              # taux de décroissance pour la moyenne des gradients
    epsilon=1e-7,          # petite constante pour la stabilité numérique
    name="Adadelta"
)

# Poids des pertes
policy_weight = 1.0
value_weight = 4.0

# Compilation du modèle
model.compile(optimizer=optimizer,
              loss={'policy': 'categorical_crossentropy', 'value': 'binary_crossentropy'},
              loss_weights={'policy': policy_weight, 'value': value_weight},
              metrics={'policy': 'categorical_accuracy', 'value': 'mse'})

# Entraînement avec scheduler
for i in range(1, epochs + 1):

    print('epoch ' + str(i))
    # Mise à jour dynamique du batch
    golois.getBatch(input_data, policy, value, end, groups, i * N)

    history = model.fit(input_data,
                        {'policy': policy, 'value': value},
                        epochs=1,
                        batch_size=batch)

    if i % 5 == 0:
        gc.collect()

    if i % epochs == 0:
        golois.getValidation(input_data, policy, value, end)
        val = model.evaluate(input_data,
                             [policy, value], verbose=0, batch_size=batch)
        print("val =", val)
        model.save('mchettih.h5')